# Проверка LLM отчетов
Тестирование формирования промптов и генерации текста на конкретных примерах.

In [ ]:
import sys
import logging
from pathlib import Path

# 1. Настройка путей и импортов
notebook_path = Path().absolute()
root_dir = notebook_path.parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from llm_report import build_report_prompt, transformers_generate, clean_llm_analysis
from rag_core.data import load_data
from rag_core.components import add_component_text
from rag_core.constants import PROJECT_ROOT

# Настройка логов
logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)

In [ ]:
# 2. Загрузка тестовых данных
df = add_component_text(load_data(PROJECT_ROOT / "Best-Buy-dataset-clean.csv"))

test_idx = 0
query = df.iloc[test_idx].to_dict()
analogs = [
    (1, 0.95, df.iloc[1].to_dict()),
    (2, 0.88, df.iloc[2].to_dict()),
]

print(f"Тестовый товар: {query.get('title')}")

In [ ]:
# 3. Сборка промпта
prompt = build_report_prompt(query_item=query, analog_items=analogs, language="ru")
print("--- СФОРМИРОВАННЫЙ ПРОМПТ ---")
print(prompt)

In [ ]:
# 4. Пробная генерация
print("--- ГЕНЕРАЦИЯ (Qwen 0.5B) ---")
raw_output = transformers_generate(prompt, model="qwen", allow_download=False)

if raw_output:
    print("\nСЫРОЙ ВЫВОД:")
    print(raw_output)
    
    clean_output = clean_llm_analysis(raw_output)
    print("\nОЧИЩЕННЫЙ ВЫВОД:")
    print(clean_output)
else:
    print("Ошибка: модель не сгенерировала текст или не найдена в кеше.")